# pyjmri quickstart — interactive

Same flow as the README quickstart, split into cells so the `Client` stays open across the whole session. Run the cells top-to-bottom once, then re-run the **flip** cell as often as you like — it commands the opposite of the current state each time. When you're done, run the **close** cell (or restart the kernel).

In [1]:
import logging
from pyjmri import Client, TurnoutState

In [2]:
logging.basicConfig(
    level=logging.DEBUG,                      # use DEBUG to see WebSocket traffic
    format="%(asctime)s %(name)s %(levelname)s %(message)s",
    filename="pyjmri.log",                   # omit to log to stderr instead
    force=True,                              # re-run-safe in Jupyter
)

## Open the client (run once)

`Client.__aenter__` opens the HTTP and WebSocket connections; the returned object stays bound to `jmri` for the rest of the notebook. Re-running this cell on an already-open client raises `RuntimeError` — close it first if you need to reconnect.

In [3]:
jmri = await Client().__aenter__()
layout = await jmri.discover()
print(f"discovered: {len(layout.turnouts)} turnouts, {len(layout.sensors)} sensors")

discovered: 52 turnouts, 63 sensors


In [7]:
  COLLECTIONS = [
      ("turnouts",     "state"),
      ("sensors",      "state"),
      ("blocks",       "state"),
      ("lights",       "state"),
      ("memories",     "value"),
      ("signal_heads", "appearance"),
      ("signal_masts", "aspect"),
      ("routes",       None),       # no readable state
  ]

  def dump_layout(layout) -> None:
      for attr, status_attr in COLLECTIONS:
          coll = getattr(layout, attr)
          print(f"\n=== {attr} ({len(coll)}) ===")
          for entity in coll.values():
              user = f" [{entity.user_name}]" if entity.user_name else ""
              if status_attr is None:
                  print(f"  {entity.name}{user}")
              else:
                  status = getattr(entity, status_attr)
                  status_str = status.name if hasattr(status, "name") else repr(status)
                  print(f"  {entity.name}{user}: {status_str}")

  dump_layout(layout)


=== turnouts (52) ===
  NT100 [South Turnout 100]: THROWN
  NT101 [South Turnout 101]: UNKNOWN
  NT102 [South Turnout 102]: CLOSED
  NT103 [South Turnout 103]: UNKNOWN
  NT104 [South Turnout 104]: UNKNOWN
  NT105 [South Turnout 105]: CLOSED
  NT106 [South Turnout 106]: CLOSED
  NT107 [South Turnout 107]: UNKNOWN
  NT108 [South Turnout 108]: UNKNOWN
  NT109 [South Turnout 109]: CLOSED
  NT110 [South Turnout 110]: CLOSED
  NT111 [South Turnout 111]: CLOSED
  NT112 [South Turnout 112]: CLOSED
  NT113 [South Turnout 113]: UNKNOWN
  NT114 [South Turnout 114]: UNKNOWN
  NT115 [South Turnout 115]: CLOSED
  NT116 [South Turnout 116]: CLOSED
  NT117 [South Turnout 117]: CLOSED
  NT118 [South Turnout 118]: CLOSED
  NT201 [East Turnout 201]: CLOSED
  NT202 [East Turnout 202]: CLOSED
  NT203 [East Turnout 203]: CLOSED
  NT204 [East Turnout 204]: CLOSED
  NT205 [East Turnout 205]: CLOSED
  NT206 [East Turnout 206]: CLOSED
  NT207 [East Turnout 207]: THROWN
  NT208 [East Turnout 208]: CLOSED
  NT20

In [6]:
{a: len(getattr(layout, a)) for a, _ in COLLECTIONS}

{'turnouts': 52,
 'sensors': 63,
 'blocks': 38,
 'lights': 0,
 'routes': 15,
 'memories': 9,
 'signal_heads': 12,
 'signal_masts': 12}

In [4]:
layout

## Pick a turnout

Grab the first turnout in the layout. To target a specific one instead, swap in `layout.turnouts.by_system_name("NT100")` or `layout.turnouts.by_user_name("North Yard Lead")`.

In [4]:
turnout = next(iter(layout.turnouts.values()))
print(f"name={turnout.name} user_name={turnout.user_name} initial state={turnout.state.name}")

name=NT100 user_name=South Turnout 100 initial state=CLOSED


## Flip it (re-run repeatedly)

Each run commands the opposite of whatever JMRI is currently reporting, so re-running this cell toggles the turnout back and forth.

In [5]:
target = TurnoutState.THROWN if turnout.state is TurnoutState.CLOSED else TurnoutState.CLOSED
await turnout.set_state(target)
print(f"final state={target.name}")

final state=THROWN


## Set a turnout to a specific state

In [15]:
await layout.turnouts.by_system_name("NT102").set_state(TurnoutState.CLOSED)


## Close the client

Run this when you're done to shut down the WebSocket and background tasks cleanly. Kernel restart also cleans up if you forget.

In [16]:
await jmri.__aexit__(None, None, None)